# Lab 0B: Compare Three Models on PII Extraction

Use this notebook after you finish `../lab0_1_environment_setup/03_environment_check.ipynb`.

In this warm-up notebook, you will use three class models from the same Ollama endpoint and compare how they extract PII and device identifiers from a short synthetic case note.

This task is a good fit for Lab 0 because it is:
- easy to run
- relevant to digital forensics
- simple enough to compare by observation before you revise a prompt
- based on synthetic data rather than real personal data

In [1]:
import json
from pathlib import Path
from time import perf_counter

import requests
from dotenv import dotenv_values
from openai import OpenAI

# Teaching note:
# This setup cell assumes you opened the notebook from this lab folder, then
# loads this lab's .env and builds the model client used in the warm-up.
LAB_NAME = 'lab0_2_model_warmup'

lab_dir = Path.cwd().resolve()
if lab_dir.name != LAB_NAME:
    raise FileNotFoundError(
        f'Open this notebook from the {LAB_NAME} folder.'
    )

repo_root = lab_dir.parent
env_example_path = lab_dir / '.env.example'
if not env_example_path.exists():
    raise FileNotFoundError(f'Expected .env.example in {LAB_NAME}.')

env_path = lab_dir / '.env'
if not env_path.exists():
    raise FileNotFoundError(
        f'Expected .env in this folder. Copy .env.example to .env first.'
    )

config = dotenv_values(env_path)
default_model = config.get('MODEL')
ollama_base_url = config.get('OLLAMA_BASE_URL')

if not default_model or not ollama_base_url:
    raise ValueError("MODEL or OLLAMA_BASE_URL is missing from this lab's .env")

client = OpenAI(base_url=ollama_base_url, api_key='ollama')

print('Repo root:', repo_root)
print('Lab folder:', lab_dir)
print("Default model from this lab's .env:", default_model)
print('Ollama endpoint:', ollama_base_url)


Repo root: /home/frank/projects/agentic-AI4-forensics
Lab folder: /home/frank/projects/agentic-AI4-forensics/lab0_2_model_warmup
Default model from this lab's .env: qwen3:8b
Ollama endpoint: http://localhost:11434/v1


## Step 1: Discover Available Models

Run the next cell to list the models available from the configured Ollama endpoint.

In [2]:
# Teaching note:
# This block asks Ollama for the available model list and a simple size lookup.
# It turns the raw server response into something easier to compare in class.
tags_url = ollama_base_url.rstrip('/').replace('/v1', '/api/tags')

response = requests.get(tags_url, timeout=10)
response.raise_for_status()

models_data = response.json().get('models', [])
available_models = []
model_size_lookup = {}

print('Available models:')
for item in models_data:
    model_name = item.get('name')
    if not model_name:
        continue

    available_models.append(model_name)
    size_bytes = item.get('size')
    if isinstance(size_bytes, (int, float)):
        size_text = f'{size_bytes / (1024 ** 3):.2f} GB'
    else:
        size_text = 'size unavailable'

    model_size_lookup[model_name] = size_text
    print(f'{len(available_models)}. {model_name} ({size_text})')

if len(available_models) < 3:
    raise ValueError('This homework needs at least 3 available models.')


Available models:
1. gemma4:31b (18.50 GB)
2. gemma4:e4b (8.95 GB)
3. huihui_ai/qwen3.5-abliterated:latest (16.22 GB)
4. qwen3.5:2b (2.55 GB)
5. qwen3.5:9b (6.14 GB)
6. qwen3.5:0.8b (0.96 GB)
7. qwen3.5:35b-a3b (22.23 GB)
8. qwen3:8b (4.87 GB)
9. qwen3.5:27b (16.22 GB)
10. llama3.3:70b (39.60 GB)


## Step 2: Choose Three Models

Use these three models for the comparison so everyone in the class works with the same set:
- `qwen3.5:0.8b`
- `qwen3.5:27b`
- `gemma4:e4b`

The next cell checks whether all three are available at the configured endpoint.

In [3]:
# Teaching note:
# This block locks the notebook to the fixed class model set.
# It also checks whether any required model is missing from the endpoint.
models_to_compare = ['qwen3.5:0.8b', 'qwen3.5:27b', 'gemma4:e4b']

missing_models = [name for name in models_to_compare if name not in available_models]

print('Models selected for comparison:')
for model_name in models_to_compare:
    size_text = model_size_lookup.get(model_name, 'size unavailable')
    print(f'- {model_name} ({size_text})')

if missing_models:
    raise ValueError(f'These required models are missing from the endpoint: {missing_models}')

if len(set(models_to_compare)) != 3:
    raise ValueError('Choose 3 different models before continuing.')


Models selected for comparison:
- qwen3.5:0.8b (0.96 GB)
- qwen3.5:27b (16.22 GB)
- gemma4:e4b (8.95 GB)


## Step 3: Synthetic Case Note

Use the same synthetic case note for all three models.

In [4]:
case_note = """
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.
""".strip()

print(case_note)

Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.


## Step 4: Shared Extraction Prompt

All three models should answer the **same prompt** so you can compare them fairly.

In [5]:
comparison_prompt = f"""
Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
{case_note}
""".strip()

print(comparison_prompt)

Extract PII from the synthetic case note below.

Return valid JSON only.
Use exactly these keys:
- names
- phone_numbers
- email_addresses
- physical_addresses
- device_ids

Rules:
- Keep each item exactly as it appears in the note.
- Use arrays for all five keys.
- Do not include explanations.
- Do not add keys beyond the five listed above.

Case note:
Case note:
Investigator Maya Chen documented an interview with Jordan Lee at 2458 West Pine Street, Springfield, IL 62704.
Jordan Lee said a suspicious text came from 415-555-0187 and referenced alex.romero88@example.com.
A second contact for follow-up was Priya Nair at 202-555-0142 and priyanair@sample.org.
The seized phone record listed IMEI 356938035643809 and serial number SN-A19XZ-4421.


### Run the Same Prompt on All Three Models

This cell defines two small helper functions and then runs the same prompt on all three selected models. The goal is to keep the task identical so you can compare the outputs fairly.


In [6]:
def clean_json_text(text: str) -> str:
    cleaned = text.strip()
    if cleaned.startswith('```'):
        cleaned = cleaned.strip('`')
        cleaned = cleaned.replace('json\n', '', 1).strip()
    start = cleaned.find('{')
    end = cleaned.rfind('}')
    if start != -1 and end != -1 and end > start:
        cleaned = cleaned[start:end + 1]
    return cleaned

def ask_model(model_name: str, prompt: str) -> dict:
    start = perf_counter()
    response = client.chat.completions.create(
        model=model_name,
        messages=[{'role': 'user', 'content': prompt}]
    )
    elapsed = perf_counter() - start
    raw_text = response.choices[0].message.content
    cleaned_text = clean_json_text(raw_text)

    try:
        parsed = json.loads(cleaned_text)
        parse_error = None
    except Exception as exc:
        parsed = None
        parse_error = str(exc)

    return {
        'model': model_name,
        'seconds': round(elapsed, 2),
        'raw_text': raw_text,
        'parsed': parsed,
        'parse_error': parse_error,
    }

results = []
for model_name in models_to_compare:
    print(f'Running {model_name}...')
    results.append(ask_model(model_name, comparison_prompt))

print('Comparison complete.')


Running qwen3.5:0.8b...
Running qwen3.5:27b...
Running gemma4:e4b...
Comparison complete.


### Inspect the Raw Outputs

Run this cell to print each model's raw response exactly as it came back from the API. Do not worry if the formats are different yet. In this warm-up, noticing those differences is part of the task.


In [7]:
for result in results:
    print('=' * 80)
    print('Model:', result['model'])
    print('Time (seconds):', result['seconds'])
    print('Parse error:', result['parse_error'])
    print('-' * 80)
    print(result['raw_text'])
    print()

Model: qwen3.5:0.8b
Time (seconds): 116.13
Parse error: None
--------------------------------------------------------------------------------
{
    "names": ["Maya Chen", "Jordan Lee", "Priya Nair", "Alex.romero88"],
    "phone_numbers": ["415-555-0187", "202-555-0142"],
    "email_addresses": ["alex.romero88@example.com", "priyanair@sample.org"],
    "physical_addresses": ["2458 West Pine Street, Springfield, IL 62704"],
    "device_ids": ["IMEI 356938035643809", "SN-A19XZ-4421"]
}

Model: qwen3.5:27b
Time (seconds): 160.51
Parse error: None
--------------------------------------------------------------------------------
{
  "names": [
    "Maya Chen",
    "Jordan Lee",
    "Priya Nair"
  ],
  "phone_numbers": [
    "415-555-0187",
    "202-555-0142"
  ],
  "email_addresses": [
    "alex.romero88@example.com",
    "priyanair@sample.org"
  ],
  "physical_addresses": [
    "2458 West Pine Street, Springfield, IL 62704"
  ],
  "device_ids": [
    "356938035643809",
    "SN-A19XZ-4421"
  

## Step 5: Build a Simple Comparison Summary

Instead of a detailed grading rubric, this step creates a small summary that is easier to read in a warm-up lab.

For each model, the summary shows:
- response time in seconds
- whether the response could be parsed as JSON
- which keys were returned
- what the model put in `device_ids`

In [8]:
comparison_summary = []

for result in results:
    parsed = result['parsed'] if isinstance(result['parsed'], dict) else None
    keys_found = sorted(parsed.keys()) if isinstance(parsed, dict) else []
    device_ids_found = parsed.get('device_ids', []) if isinstance(parsed, dict) else []

    comparison_summary.append({
        'model': result['model'],
        'seconds': result['seconds'],
        'valid_json': parsed is not None,
        'keys_found': keys_found,
        'device_ids_found': device_ids_found,
    })

print(json.dumps(comparison_summary, indent=2))

[
  {
    "model": "qwen3.5:0.8b",
    "seconds": 116.13,
    "valid_json": true,
    "keys_found": [
      "device_ids",
      "email_addresses",
      "names",
      "phone_numbers",
      "physical_addresses"
    ],
    "device_ids_found": [
      "IMEI 356938035643809",
      "SN-A19XZ-4421"
    ]
  },
  {
    "model": "qwen3.5:27b",
    "seconds": 160.51,
    "valid_json": true,
    "keys_found": [
      "device_ids",
      "email_addresses",
      "names",
      "phone_numbers",
      "physical_addresses"
    ],
    "device_ids_found": [
      "356938035643809",
      "SN-A19XZ-4421"
    ]
  },
  {
    "model": "gemma4:e4b",
    "seconds": 22.91,
    "valid_json": true,
    "keys_found": [
      "device_ids",
      "email_addresses",
      "names",
      "phone_numbers",
      "physical_addresses"
    ],
    "device_ids_found": [
      "IMEI 356938035643809",
      "SN-A19XZ-4421"
    ]
  }
]


## Reflection Questions

Replace this text with short answers to the questions below.

Use the summary above and the raw outputs to support your answers.

1. Which model was the fastest?
2. Which models returned valid JSON?
3. Did the three models use the same `device_ids` format? Describe one difference you noticed.
4. Which response was easiest to read and why?
5. What is one prompt rule you might add before doing `03_prompt_revision_assignment.ipynb`?

## Submission

Save the notebook with:
- the discovered model list
- the three selected models
- the raw outputs from all three models
- the simple comparison summary
- your short reflection